# Flask Application Code

In [ ]:
#Importing Libraries
from flask import Flask, render_template, request, redirect, url_for, flash, session, redirect
import logging, re, os
from datetime import datetime
from werkzeug.security import generate_password_hash, check_password_hash 
import sqlite3
import csv
import math
import ast
import random
import os

from openpyxl import load_workbook
#from faker import Faker
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import string
import random
import csv

#Libraries for machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

#Set Seed
SEED_VAL = 55
random.seed(SEED_VAL)
np.random.seed(SEED_VAL)

# ------------------------------------
#Initialising Flask Application
app = Flask(__name__)
# ------------------------------------

# ------------------------------------
# In production set SECRET_KEY via environment variable
app.secret_key = os.environ.get("SECRET_KEY", "ftgongvsbn7283")

# ------------------------------------

# ------------------------------------
# Database paths
DB_PATH = "patientdb.db"
CSV_PATH = "cleandata.csv"
# ------------------------------------

# ------------------------------------
#Creating patient table if they don't exist
def init_db(CSV_PATH):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    #Creating the table structure
    #Email needs to be unique so it is a primary key
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS patientdata (
    "Age"	INTEGER,
	"Height_cm"	REAL,
	"Weight_kg"	REAL,
	"BMI"	REAL,
	"Menstrual_Cycle_Length_days"	INTEGER,
	"Menstrual_Irregularity"	INTEGER,
	"Fasting_Glucose_mg_dL"	REAL,
	"Fasting_Insulin_uIU_mL"	REAL,
	"HOMA_IR"	REAL,
	"LH_mIU_mL"	REAL,
	"FSH_mIU_mL"	REAL,
	"LH_FSH_Ratio"	REAL,
	"Total_Testosterone_ng_dL"	REAL,
	"Free_Testosterone_pg_mL"	REAL,
	"Total_Cholesterol_mg_dL"	INTEGER,
	"Triglycerides_mg_dL"	REAL,
	"Dietary_Sugar_Intake"	INTEGER,
	"Physical_Activity_Level"	INTEGER,
	"PCOS_Diagnosis"	INTEGER,
	"Hirsutism_Score_FG"	INTEGER,
	"Acne_Severity"	INTEGER,
	"Alopecia"	INTEGER,
	"Skin_Darkening_Acanthosis"	INTEGER,
	"first_name"	TEXT,
	"last_name"	TEXT,
	"email"	TEXT PRIMARY KEY,
	"password"	TEXT,
	"Target_Nutrient_Vector"	TEXT
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """)
    conn.commit()

    #Checking if the CSV file path exists before reading
    if not os.path.exists(CSV_PATH):
        print(f"Error: {CSV_PATH} not found.")
        return
    
    #Checking if table is empty before populating to avoid duplication
    cursor.execute("SELECT COUNT(*) FROM patientdata")
    if cursor.fetchone()[0] == 0:

        with open(CSV_PATH, mode='r', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            for row in reader:
                #Applying hashing encryption to passwords
                hashed_pw = generate_password_hash(row['password'], method="pbkdf2:sha256")

                #Inserting the rows from the CSV file
                cursor.execute("""
                    INSERT OR REPLACE INTO patientdata (
                        Age, Height_cm, Weight_kg, BMI, 
                        Menstrual_Cycle_Length_days, Menstrual_Irregularity,
                        Fasting_Glucose_mg_dL, Fasting_Insulin_uIU_mL,
                        HOMA_IR, LH_mIU_mL, FSH_mIU_mL, LH_FSH_Ratio,
                        Total_Testosterone_ng_dL, Free_Testosterone_pg_mL,
                        Total_Cholesterol_mg_dL, Triglycerides_mg_dL,
                        Dietary_Sugar_Intake, Physical_Activity_Level,
                        PCOS_Diagnosis, Hirsutism_Score_FG, Acne_Severity,
                        Alopecia, Skin_Darkening_Acanthosis, first_name, 
                        last_name, email, password, Target_Nutrient_Vector,
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    """, (

                        row['Age'], row['Height_cm'], row['Weight_kg'], row['BMI'],
                        row['Menstrual_Cycle_Length_days'], row['Menstrual_Irregularity'],
                        row['Fasting_Glucose_mg_dL'], row['Fasting_Insulin_uIU_mL'],
                        row['HOMA_IR'], row['LH_mIU_mL'], row['FSH_mIU_mL'], row['LH_FSH_Ratio'],
                        row['Total_Testosterone_ng_dL'], row['Free_Testosterone_pg_mL'],
                        row['Total_Cholesterol_mg_dL'], row['Triglycerides_mg_dL'],
                        row['Dietary_Sugar_Intake'], row['Physical_Activity_Level'],
                        row['PCOS_Diagnosis'], row['Hirsutism_Score_FG'], row['Acne_Severity'],
                        row['Alopecia'], row['Skin_Darkening_Acanthosis'], row['first_name'],
                        row['last_name'], row['email'], hashed_pw, row['Target_Nutrient_Vector']
                    ))
        conn.commit()
        conn.close()
        print("CSV data and hashed passwords loaded successfully.")


# ------------------------------------
#Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler("app.log"), logging.StreamHandler()],
)
log = logging.getLogger(__name__)
# ------------------------------------

# ------------------------------------
# Validation patterns

##Ensure that first name is only letters and hyphens
FIRSTNAME_PATTERN = re.compile(r'^[A-Za-z-]{1,50}$')

##Ensure that last name is only letters and hyphens
LASTNAME_PATTERN = re.compile(r'[A-Za-z-]{1,50}$')

## Ensures the email has a basic valid structure of name@domain.tld
EMAIL_PATTERN = re.compile(r'^[\w\.-]+@[\w\.-]+\.[A-Za-z]{2,}$')

## Strong password pattern that requires lowercase, uppercase, digit, special character, and minimum 8 characters.
PASSWORD_PATTERN = re.compile(r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$')
# ------------------------------------

# ------------------------------------
# Calculating and viewing user recommendations 

def user_recs(user_email, db_path="patientdb.db", top_n=10):
    #Establishing database connection
    conn = sqlite3.connect(db_path)

    #Query the database for the user's target nutrient vector
    user_query = """
        SELECT Target_Nutrient_Vector
        FROM patientdata
        WHERE lower(email) = lower(?)
    """
    cursor = conn.cursor()
    cursor.execute(user_query, (user_email,))
    user_row = cursor.fetchone()

    if not user_row:
        conn.close()
        return None
    
    #Retrieving patient nutrient vector
    #Converting the nutrient vector TEXT string into a real list
    patient_vector = user_row[0]
    patient_vector = ast.literal_eval(patient_vector)


    #Loading the saved food_matrix_5d table 
    food_df = pd.read_sql_query("SELECT * FROM food_matrix_5d", conn)

    #Renaming the Raw DQL nutrient column names
    food_df = food_df.rename(columns={
        "Fiber, total dietary": 'fiber_g',
        "Fatty acids, total polyunsaturated": 'pufa_g',
        "Magnesium, Mg": 'magnesium_mg',
        "Vitamin_D_Total_UG": 'vitamin_d_mcg',
        "Zinc, Zn": 'zinc_mg'
    })

    #Retrieving the names of foods to attach them to the matrix
    names_df = pd.read_sql_query("SELECT description AS food_description FROM food", conn)
    food_df['food_description'] = names_df['food_description']

    conn.close()

    #Isolating the 5 nutrients in the vector
    #For scaling
    nutrient_cols = ['fiber_g', 'pufa_g', 'magnesium_mg', 'vitamin_d_mcg', 'zinc_mg']
    
    #Mapping the nutrient values to the food matrix
    food_matrix = food_df[nutrient_cols].values
    patient_matrix = np.array(patient_vector).reshape(1, -1)

    #Scaling the nutrients
    scaler = MinMaxScaler()
    scaled_foods = scaler.fit_transform(food_matrix)
    scaled_patient = scaler.transform(patient_matrix)

    #Running the cosine similarity engine 
    similarity_scores = cosine_similarity(scaled_patient, scaled_foods)[0]
    food_results_df = food_df.copy()
    food_results_df['Match_Score'] = np.round(similarity_scores * 100, 1)

    #Returning the top N items as a list of dictionaries for Jinja2 HTML rendering
    top_foods = food_results_df.sort_values(by='Match_Score', ascending=False).head(top_n)
    return top_foods.to_dict(orient='records')
    

# ------------------------------------
# Routes for pages

# Route for the Home page
@app.route('/')
def home():
    return render_template('home.html')

# Route for the About page
@app.route('/about')
def about():
    return render_template('about.html')

# Route for the login page
@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == "GET":
        return render_template("login.html")
    
    # ----- Post request handling -----
    email = request.form.get("email", "").strip()
    password = request.form.get("password", "")

    if not (email and password):
        flash("Please enter email and password.")
        return redirect(url_for("login"))

    # ----- Database connection and Query -----
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    #Querying the user table to find a user matching the email
    cursor.execute("""
        SELECT first_name, last_name, email, password
        FROM patientdata
        WHERE lower(email) = lower(?)
    """, (email,))
    row = cursor.fetchone()
    conn.close() #Close connection immediately after fetching data

    # ----- Check if we found a user in the database
    if not row:
        flash("Invalid email or password.")
        return redirect(url_for("login"))
    
    #Extracting the user data from the database row
    db_first_name, db_last_name, db_email, db_password = row

    #Verifying password
    if password != db_password:
        flash("Invalid email or password.")
        return redirect(url_for("login"))

    #Saving user information in session upon successful login
    session["ID"] = db_email
    session["username"] = f"{db_first_name} {db_last_name}"
    return redirect(url_for("dashboard"))

# Route for the patient dashboard
@app.route("/dashboard")
def dashboard():
    if "ID" not in session:
        flash("Please log in to continue.")
        return redirect(url_for("login"))
    return render_template("dashboard.html")

# Route for the register page
@app.route('/register', methods =['GET', 'POST'])
def register():
    if request.method == 'POST':
        First_name = (request.form.get("first-name") or "").strip()
        Last_name = (request.form.get("surname") or "").strip()
        email    = (request.form.get("email") or "").strip()
        password = (request.form.get("password") or "").strip()
        
        try:

            ## ---- SERVER-SIDE INPUT VALIDATION FOR REGISTRATION FORM ---- ##

            # ----- Checking empty fields -----
             if not First_name:
                 raise ValueError("First name is required")
             if not Last_name:
                 raise ValueError("Last name is required")
             if not email:
                 raise ValueError("Email is required.")
             if not password:
                 raise ValueError("Password is required.")
             
            # ----- Type / Format checks -----
             if not FIRSTNAME_PATTERN.fullmatch(First_name):
                  raise ValueError("First name must only contain letters and hyphens")

             if not LASTNAME_PATTERN.fullmatch(Last_name):
                  raise ValueError("Last name must only contain letters and hyphens")
             
             if not EMAIL_PATTERN.fullmatch(email):
                  raise ValueError("Email format is invalid.")
             
             if not PASSWORD_PATTERN.fullmatch(password):
                  raise ValueError("Password format is invalid.")
             
             if len(email) > 254:
                  raise ValueError("Email too long.")

            # ----- Hashing password ------
             hashed_password = generate_password_hash(password, method="pbkdf2:sha256")

             conn= sqlite3.connect(DB_PATH)
             cursor = conn.cursor()

            # ----- Pre-check duplicate -----
             cursor.execute("SELECT 1 FROM patientdata WHERE lower(email) = lower(?)", (email,))
             if cursor.fetchone():
                 flash("This email is already registered. Please log in instead.")
                 conn.close()
                 return redirect(url_for("login"))
             
            # ----- Insert the user into the database ----- 
             try:
                 cursor.execute("""
                    INSERT INTO patientdata (first_name, last_name, email, password)
                    VALUES (?, ?, ?, ?)
                 """, (First_name, Last_name, email, hashed_password)) #doctor is set as default role for role based access 
                 conn.commit()
                 flash("Registration successful! Please log in.")
             except sqlite3.IntegrityError: 
                 flash("This email is already registered. Please log in.")
             finally:
                 conn.close()

             return redirect(url_for("success"))
        
        except ValueError as e:
            flash(str(e), 'error')
            log.warning("Validation failed: %s", e)
            return redirect(url_for("register"))
       
    return render_template("register.html")

# Route for Registration success page 
@app.route('/success')
def success():
    return render_template('register_success.html')

# Route for the view recommendations page
@app.route('/view_recs')
def view_recs():
    if "ID" not in session:
        flash("Please log in to view your recommendations.") 
        return redirect(url_for("login"))
    
    user_email = session["ID"]

    #Running the recommender engine
    top_10_foods = user_recs(user_email, DB_PATH, top_n=10)

    if top_10_foods is None:
        flash("Could not locate nutrient profile for your account.")
        return redirect(url_for("dashboard"))
    
    #Passing the recommendations results into the HTML page for display
    return render_template('view_recs.html', recommendations=top_10_foods, user_name=session.get("userFirstName", "User"))

# Route for the Questionnaire page
@app.route('/questionnaire', methods=['GET', 'POST'])
def questionnaire():
    if "ID" not in session:
        flash("Please log in to continue.")
        return redirect(url_for("login"))
    
    if request.method == 'POST':
        age = (request.form.get("age") or "").strip()
        height = (request.form.get("height") or "").strip()
        weight = (request.form.get("weight") or "").strip()
        cycle_length = (request.form.get("cycle_length") or "").strip()
        sugar_intake = (request.form.get("sugar_intake") or "").strip()
        physical_activity = (request.form.get("physical_activity") or "").strip()
        acne_severity = (request.form.get("acne_severity") or "").strip()
        fg_score = (request.form.get("fg_score") or "").strip()

        #Function to convert blank string to None (NULL) for optional fields
        def parse_optional(val):
            return float(val) if val.strip() else None

        #Extracting values from optional fields using the parse_optional function
        lh_level = parse_optional(request.form.get("lh_level" or ""))
        fsh_level = parse_optional(request.form.get("fsh_level" or ""))
        glucose_level = parse_optional(request.form.get("glucose_level" or ""))
        insulin_level = parse_optional(request.form.get("insulin_level" or ""))
        total_tes = parse_optional(request.form.get("total_tes" or ""))
        free_tes = parse_optional(request.form.get("free_tes" or ""))
        cholesterol = parse_optional(request.form.get("cholesterol" or ""))
        triglycerides = parse_optional(request.form.get("triglycerides" or ""))

        #Mapping yes/no radio button answers to 1/0 for the database
        menstrual_irregularity = 1 if request.form.get("menstrual_irregularity") == "Yes" else 0
        alopecia = 1 if request.form.get("alopecia") == "Yes" else 0
        acanthosis = 1 if request.form.get("acanthosis") == "Yes" else 0
        hirsutism = 1 if request.form.get("hirsutism") == "Yes" else 0

        #Calculation for the BMI
        try:
            height_m = float(height) / 100
            bmi = round(float(weight) / (height_m * height_m), 1)
        except ValueError:
            bmi = None

        #Calculation for the HOMA IR level
        try:
            homa_ir = (insulin_level / glucose_level) / 405
        except ValueError:
            homa_ir = None

        #Calculation for LH/FSH Ratio
        try: 
            lh_fsh_ratio = lh_level / fsh_level
        except ValueError: 
            lh_fsh_ratio = None

        #Using the session ID (user email) to make sure the data is inserted to the record of the logged in user
        user_email = session["ID"]

        try:

            ## ---- SERVER-SIDE INPUT VALIDATION FOR REGISTRATION FORM ---- ##

            # ----- Checking empty fields -----
             if not age:
                 raise ValueError("Age is required")
             if not height:
                 raise ValueError("Height is required")
             if not weight:
                 raise ValueError("Weight is required.")
             if not cycle_length:
                 raise ValueError("Cycle length is required.")
             if not sugar_intake:
                 raise ValueError("Sugar intake is required.")
             if not physical_activity:
                 raise ValueError("Physical activity level is required.")
             if not physical_activity:
                 raise ValueError("Physical activity level is required.")
             if not acne_severity:
                 raise ValueError("Acne severity is required.")
             if not fg_score:
                 raise ValueError("Ferriman-Gallwey Hirsutism is required.")
             
            # ----- Inserting the data into the database -----
             try:
                 conn = sqlite3.connect(DB_PATH)
                 cursor = conn.cursor()

                 #SQL query to UPDATE the existing user's row with the new data
                 cursor.execute("""
                    UPDATE patientdata
                    SET Age = ?, Height_cm = ?, Weight_kg = ?, BMI = ?, Menstrual_Cycle_Length_days = ?,
                                Menstrual_Irregularity = ?, Dietary_Sugar_Intake = ?, Physical_Activity_Level = ?,
                                Acne_Severity = ?, Alopecia = ?, Skin_Darkening_Acanthosis = ?, Hirsutism_Score_FG = ?,
                                LH_mIU_mL = ?, FSH_mIU_mL = ?, LH_FSH_Ratio = ?, Fasting_Glucose_mg_dL = ?, Fasting_Insulin_uIU_mL = ?,
                                HOMA_IR = ?, Total_Testosterone_ng_dL = ?, Free_Testosterone_pg_mL = ? , Total_Cholesterol_mg_dL = ?, Triglycerides_mg_dL = ?
                    WHERE email = ?
                 """, (
                    age, height, weight, bmi, cycle_length, 
                    menstrual_irregularity, sugar_intake, physical_activity,
                    acne_severity, alopecia, acanthosis, fg_score, 
                    lh_level, fsh_level, lh_fsh_ratio, glucose_level, 
                    insulin_level, homa_ir, total_tes, free_tes, 
                    cholesterol, triglycerides, user_email
                 ))  
             
                 conn.commit()
                 flash("Your health profile was updated successfully.")
                 return redirect(url_for("dashboard"))
        
             except Exception as e:
                flash(f"An error occured while saving: {str(e)}", "error")
                log.warning("Database error during questionnaire update: %s", e)
             finally:
                conn.close()

        except ValueError as e:
            flash(str(e), 'error')
            log.warning("Validation failed: %s", e)
            return redirect(url_for("questionnaire"))

    return render_template('questionnaire.html')

if __name__ == "__main__":
    init_db()
    populate_db(CSV_PATH)
    app.run(debug=False)  